# 02 · Chạy baseline qua script dùng chung — Qwen2.5-VL-3B (4-bit)

**Mục tiêu:** chạy `src/eval/run_baseline.py` cho một dataset (mặc định **ViTextVQA chính thức, tập con 2.000 câu test, seed 42**) và xem kết quả. Mọi logic (nạp model theo config, chạy theo lô, resume, gộp, tính metric, ghi log) nằm trong script — notebook chỉ điều phối.

**Điều kiện:** Runtime → Change runtime type → **T4 GPU**. Data đã chuẩn bị sẵn trên Drive (notebook `00`).

**Nếu Colab rớt:** chạy lại từ phần 1 — các lô đã xong tự bỏ qua.

## 1. Thiết lập môi trường

Mount Drive, clone/pull code mới nhất, cài thư viện (pin `transformers==5.17.0` như baseline ViVQA để nhất quán), nạp `HF_TOKEN` từ Secrets nếu có (chỉ cần khi zip ảnh chưa có trên Drive).

*Kỳ vọng: in commit (≥ `ae1cf6d`), GPU `Tesla T4`, và `✅ Thiết lập xong`. ~1–2 phút.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')

!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes huggingface_hub pyyaml

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (ổn nếu zip ảnh đã nằm trên Drive)')

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU -> đổi runtime sang T4')
print('✅ Thiết lập xong')

## 2. Chọn dataset và tên run

Đổi `DATASET` nếu muốn chạy dataset khác. `RUN_NAME` là tên thư mục kết quả trong `experiments/` trên Drive — theo quy ước `W0X_<chế độ>_<model>_<dataset>_seed42`. **Không dùng lại tên run cũ** cho dữ liệu khác (script sẽ báo lỗi để tránh trộn kết quả).

*Kỳ vọng: in ra dataset và tên run.*

In [ ]:
DATASET = 'vitextvqa_official'        # 'vitextvqa_official' | 'vivqa'
RUN_NAMES = {
    'vitextvqa_official': 'W05_zeroshot_qwen25vl_vitextvqa_test2000_seed42',
    'vivqa':              'W05_zeroshot_qwen25vl_vivqa_test_seed42',   # đã có kết quả chính thức
}
RUN_NAME = RUN_NAMES[DATASET]
CHUNK = 500
print('Dataset :', DATASET)
print('Run name:', RUN_NAME)

## 3. Chạy baseline (theo lô, resume được)

Script tự: kiểm tra config đúng bản chuẩn W05 → đưa ảnh về `/content` (ViTextVQA: giải nén zip từ Drive, ~1,5 phút) → nạp model (~4 phút) → chạy từng lô 500 câu, lưu ngay lên Drive → gộp và tính EM / VQA-Acc / ANLS → ghi `metrics.json` + `predictions.json`.

*Kỳ vọng: 4 lô, dòng `[lô k/4] xong ...` hiện dần; tổng ~35–45 phút. Giữ tab mở.*

In [ ]:
!python -u -m src.eval.run_baseline --dataset {DATASET} --run_name {RUN_NAME} --chunk {CHUNK}

## 4. Xem kết quả

Đọc `metrics.json` (điểm + thông tin tái lập) và in 15 ví dụ ngẫu nhiên (seed 0) để soi nhanh dự đoán so với đáp án.

*Kỳ vọng: in EM / VQA-Acc / ANLS, thời gian, GPU, commit, và 15 cặp PRED/GOLD.*

In [ ]:
import json, random
OUT = f'/content/drive/MyDrive/ViVQA-VLM/experiments/{RUN_NAME}'
res = json.load(open(f'{OUT}/metrics.json', encoding='utf-8'))
m, meta = res['metrics'], res['meta']
print(f"==== {RUN_NAME} ====")
for k in ('exact_match', 'vqa_accuracy', 'anls'):
    print(f'{k:14s}: {m[k]:.4f}')
print(f"mẫu: {meta['num_samples']} | {meta['inference_time_sec']}s ({meta['sec_per_sample']} s/mẫu)")
print(f"GPU: {meta['hardware']['gpus_seen']} | commit {meta['git_commit']} | max_pixels {meta['max_pixels']}")

preds = json.load(open(f'{OUT}/predictions.json', encoding='utf-8'))
print('\n==== 15 ví dụ ngẫu nhiên ====')
for r in random.Random(0).sample(preds, 15):
    print(f"Q: {r['question']}\n   PRED: {r['prediction']!r}\n   GOLD: {r['answers']}\n")